In [1]:
import os
import json

In [2]:
def list_config_files():
    dir = "../vsomeip-configs"
    if not os.path.exists(dir):
        print(f"Directory {dir} does not exist.")
        return
    files = [f for f in os.listdir(dir) if f.endswith('.json')]
    if not files:
        print("No JSON configuration files found.")
        return
    return files

def read_all_configs():
    dir = "../vsomeip-configs"
    files = list_config_files()
    if not files:
        return

    configs = {}
    for file in files:
        with open(os.path.join(dir, file), 'r') as f:
            content = json.load(f)
            configs[file.split(".")[0]] = content
    return configs

configs = read_all_configs()

In [3]:
# list subfolders in current directory
def list_subfolders():
    # current_directory = os.getcwd()
    # subfolders = [f.name for f in os.scandir(current_directory) if f.is_dir()]
    # return subfolders
    return ["h"]

def list_log_files_in_folder(folder, must_end_with=".log"):
    if not os.path.exists(folder):
        return f"folder '{folder}' does not exist."
    
    files = [f.name for f in os.scandir(folder) if f.is_file() and (not must_end_with or f.name.endswith(must_end_with))]
    return files

svcb_queries = dict()
def parse_logs(logfile):
    if not os.path.exists(logfile):
        return f"Log file '{logfile}' does not exist."
    
    with open(logfile, 'r') as file:
        logs = file.readlines()
    
    parsed_logs = {
        "svcb_querys": 0,
        "svcb_replies": 0,
        "tlsa_querys": 0,
        "tlsa_replies": 0,
        "offer_recv": 0,
        "offer_send": 0,
        "subscribe_send": 0,
        "subscribe_recv": 0,
        "subscribe_ack_send": 0,
        "subscribe_ack_recv": 0
    }
    offer_recv = dict()
    offer_send = dict()
    subscribe_send = dict()
    subscribe_recv = dict()
    subscribe_ack_recv = dict()
    subscribe_ack_send = dict()
    for line in logs:
        if "SVCB SERVICE REQUEST SEND" in line:
            service = line.split(" ")[-1].strip()
            parsed_logs["svcb_querys"] += 1
            if logfile not in svcb_queries:
                svcb_queries[logfile] = []
            svcb_queries[logfile].append(service)
        elif "SVCB SERVICE RESPONSE RECEIVE" in line:
            parsed_logs["svcb_replies"] += 1
        elif "TLSA SERVICE REQUEST SEND" in line or "TLSA CLIENT REQUEST SEND" in line:
            parsed_logs["tlsa_querys"] += 1
        elif "TLSA SERVICE RESPONSE RECEIVE" in line or "TLSA CLIENT RESPONSE RECEIVE" in line:
            parsed_logs["tlsa_replies"] += 1
        elif "OFFER RECEIVE" in line:
            service = line.split(" ")[-1].strip()
            offer_recv[service] = 1
        elif "OFFER SEND " in line:
            service = line.split(" ")[-1].strip()
            offer_send[service] = 1
        elif "SUBSCRIBE SEND " in line:
            service = line.split(" ")[-1].strip()
            subscribe_send[service] = 1
        elif "SUBSCRIBE RECEIVE " in line:
            service = line.split(" for service ")[-1].split(" ")[0].strip()
            client = line.split(" ")[-1].strip()
            subscribe_recv[service + client] = 1
        elif "SUBSCRIBE ACK SEND " in line:
            service = line.split(" for service ")[-1].split(" ")[0].strip()
            client = line.split(" ")[-1].strip()
            subscribe_ack_send[service + client] = 1
        elif "SUBSCRIBE ACK RECEIVE " in line:
            service = line.split(" ")[-1].strip()
            subscribe_ack_recv[service] = 1            
    parsed_logs["offer_recv"] = len(offer_recv)
    parsed_logs["offer_send"] = len(offer_send)
    parsed_logs["subscribe_send"] = len(subscribe_send)
    parsed_logs["subscribe_recv"] = len(subscribe_recv)
    parsed_logs["subscribe_ack_send"] = len(subscribe_ack_send)
    parsed_logs["subscribe_ack_recv"] = len(subscribe_ack_recv)
    return parsed_logs

In [4]:
subfolders = list_subfolders()
hostlogs = dict()
for subfolder in subfolders:
    files = list_log_files_in_folder(subfolder)
    if isinstance(files, str):
        print(files)  # Print error message if subfolder does not exist
    else:
        hostlogs[subfolder] = files
hostlogs


{'h': ['conp.log',
  'zcfls.log',
  'zcrlp.log',
  'cfp.log',
  'zcfrs.log',
  'lrrp.log',
  'zcfrp.log',
  'lflp.log',
  'crp.log',
  'zcrrp.log',
  'lrlp.log',
  'zcflp.log',
  'zcrls.log',
  'zcrrs.log',
  'infp.log',
  'lfrp.log',
  'adass.log',
  'infs.log']}

In [5]:
parsed_logs = dict()
for scenario in hostlogs:
    parsed_logs[scenario] = dict()
    print(f"Scenario: {scenario}")
    for logfile in hostlogs[scenario]:
        logfile_path = os.path.join(scenario, logfile)
        parsed_logs[scenario][logfile] = parse_logs(logfile_path)

        print (f"Parsed logs for {logfile}:")
        for key, value in parsed_logs[scenario][logfile].items():
            print(f"{key}: {value}")
        print("\n")
        print("-" * 40)
        print("\n")

Scenario: h
Parsed logs for conp.log:
svcb_querys: 0
svcb_replies: 0
tlsa_querys: 1
tlsa_replies: 1
offer_recv: 0
offer_send: 1
subscribe_send: 0
subscribe_recv: 1
subscribe_ack_send: 1
subscribe_ack_recv: 0


----------------------------------------


Parsed logs for zcfls.log:
svcb_querys: 116
svcb_replies: 108
tlsa_querys: 107
tlsa_replies: 105
offer_recv: 38
offer_send: 0
subscribe_send: 14
subscribe_recv: 0
subscribe_ack_send: 0
subscribe_ack_recv: 14


----------------------------------------


Parsed logs for zcrlp.log:
svcb_querys: 0
svcb_replies: 0
tlsa_querys: 12
tlsa_replies: 12
offer_recv: 0
offer_send: 7
subscribe_send: 0
subscribe_recv: 12
subscribe_ack_send: 12
subscribe_ack_recv: 0


----------------------------------------


Parsed logs for cfp.log:
svcb_querys: 0
svcb_replies: 0
tlsa_querys: 1
tlsa_replies: 1
offer_recv: 0
offer_send: 1
subscribe_send: 0
subscribe_recv: 1
subscribe_ack_send: 1
subscribe_ack_recv: 0


----------------------------------------


Parsed l

In [6]:
for scenario in parsed_logs:
    print(f"Scenario: {scenario}")
    for parsed in parsed_logs[scenario]:
        print(f"Parsed log for {parsed}:")
        logfile_path = os.path.join(scenario, parsed)
        numApps = len(configs[parsed.split(".")[0]]["applications"])
        numClients = len(configs[parsed.split(".")[0]]["clients"])
        numServices = len(configs[parsed.split(".")[0]]["services"])
        numRemoteClients = 0
        for services in configs[parsed.split(".")[0]]["services"]:
            numRemoteClients += len(services["client-certificates"])
        # some sanity checks
        if "h" in scenario:
            if parsed_logs[scenario][parsed]["svcb_querys"] != numClients:
                print(f"Warning: Mismatch in svcb_querys ({parsed_logs[scenario][parsed]['svcb_querys']}) and expected ({numClients}) for {scenario}/{parsed}")
            if parsed_logs[scenario][parsed]["svcb_querys"] != parsed_logs[scenario][parsed]["svcb_replies"]:
                print(f"Warning: Mismatch in svcb_querys ({parsed_logs[scenario][parsed]['svcb_querys']}) and svcb_replies ({parsed_logs[scenario][parsed]['svcb_replies']}) for {scenario}/{parsed}")
            if parsed_logs[scenario][parsed]["tlsa_querys"] != parsed_logs[scenario][parsed]["tlsa_replies"]:
                print(f"Warning: Mismatch in tlsa_querys ({parsed_logs[scenario][parsed]['tlsa_querys']}) and tlsa_replies ({parsed_logs[scenario][parsed]['tlsa_replies']}) for {scenario}/{parsed}")
            if parsed_logs[scenario][parsed]["tlsa_querys"] != numClients + numRemoteClients:
                print(f"Warning: Mismatch in tlsa_querys ({parsed_logs[scenario][parsed]['tlsa_querys']}) and expected ({numClients + numRemoteClients}) for {scenario}/{parsed}")
            if parsed_logs[scenario][parsed]["offer_recv"] != parsed_logs[scenario][parsed]["svcb_querys"]:
                print(f"Warning: Mismatch in offer_recv ({parsed_logs[scenario][parsed]['offer_recv']}) and svcb_querys ({parsed_logs[scenario][parsed]['svcb_querys']}) for {scenario}/{parsed}")
            if parsed_logs[scenario][parsed]["svcb_replies"] != parsed_logs[scenario][parsed]["subscribe_send"]:
                print(f"Warning: Mismatch in svcb_replies ({parsed_logs[scenario][parsed]['svcb_replies']}) and subscribe_send ({parsed_logs[scenario][parsed]['subscribe_send']}) for {scenario}/{parsed}")
        if parsed_logs[scenario][parsed]["offer_send"] != numServices:
            print(f"Warning: Mismatch in offer_send ({parsed_logs[scenario][parsed]['offer_send']}) and expected ({numServices}) for {scenario}/{parsed}")
        if parsed_logs[scenario][parsed]["offer_recv"] != numClients:
            print(f"Warning: Mismatch in offer_recv ({parsed_logs[scenario][parsed]['offer_recv']}) and expected ({numClients}) for {scenario}/{parsed}")
        if parsed_logs[scenario][parsed]["subscribe_send"] != numClients:
            print(f"Warning: Mismatch in subscribe_send ({parsed_logs[scenario][parsed]['subscribe_send']}) and expected ({numClients}) for {scenario}/{parsed}")
        if parsed_logs[scenario][parsed]["subscribe_recv"] != parsed_logs[scenario][parsed]["subscribe_ack_send"]:
            print(f"Warning: Mismatch in subscribe_recv ({parsed_logs[scenario][parsed]['subscribe_recv']}) and subscribe_ack_send ({parsed_logs[scenario][parsed]['subscribe_ack_send']}) for {scenario}/{parsed}")
        if parsed_logs[scenario][parsed]["subscribe_recv"] != numRemoteClients:
            print(f"Warning: Mismatch in subscribe_recv ({parsed_logs[scenario][parsed]['subscribe_recv']}) and expected ({numRemoteClients}) for {scenario}/{parsed}")
        if parsed_logs[scenario][parsed]["subscribe_ack_recv"] != numClients:
            print(f"Warning: Mismatch in subscribe_ack_recv ({parsed_logs[scenario][parsed]['subscribe_ack_recv']}) and expected ({numClients}) for {scenario}/{parsed}")
    print("-" * 40)

Scenario: h
Parsed log for conp.log:
Parsed log for zcfls.log:
Parsed log for zcrlp.log:
Parsed log for cfp.log:
Parsed log for zcfrs.log:
Parsed log for lrrp.log:
Parsed log for zcfrp.log:
Parsed log for lflp.log:
Parsed log for crp.log:
Parsed log for zcrrp.log:
Parsed log for lrlp.log:
Parsed log for zcflp.log:
Parsed log for zcrls.log:
Parsed log for zcrrs.log:
Parsed log for infp.log:
Parsed log for lfrp.log:
Parsed log for adass.log:
Parsed log for infs.log:
----------------------------------------
